In [ ]:
# === BACKEND SETUP (qiskit_backend_config) ===
# -----------------------------------------------
# Common backend helper — works with Qiskit 2.x / qiskit-aer / qiskit-ibm-runtime.
# Provides drop-in replacements for the removed Aer, BasicAer, execute(), and IBMQ.
#
# HOW TO SELECT A BACKEND
# -----------------------
# Simulator (default, runs locally — no credentials needed):
#   BACKEND_NAME = "qasm_simulator"        # shot-based measurements
#   BACKEND_NAME = "statevector_simulator" # full statevector
#   BACKEND_NAME = "unitary_simulator"     # full unitary matrix
#
# Real IBM hardware (requires IBM Quantum account):
#   Set the environment variable:  IBM_QUANTUM_TOKEN=<your_token>
#   Then choose a device, e.g.:
#   BACKEND_NAME = "ibm_brisbane"
#   BACKEND_NAME = "least_busy"  # auto-pick least-busy device
#
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from qiskit_backend_config import (get_backend, execute, Aer, BasicAer,
                                   IBMQ, least_busy, job_monitor, assemble, combine)

# ── Choose your backend here ──────────────────────────────────────────────
BACKEND_NAME = "qasm_simulator"  # change to a device name for real hardware
# ── (Optional) IBM Quantum token — leave empty to use env var or saved creds
IBM_QUANTUM_TOKEN = os.environ.get("IBM_QUANTUM_TOKEN", "")

# Build the backend object used throughout this notebook
_token_arg = IBM_QUANTUM_TOKEN if IBM_QUANTUM_TOKEN else None
backend = get_backend(BACKEND_NAME, token=_token_arg)
print(f'Backend ready: {backend}')

In [ ]:
import time
from typing import List, Optional, Any
import numpy as np
import matplotlib.pyplot as plt

from docplex.mp.model import Model

from qiskit_backend_config import BasicAer
from qiskit_optimization.minimum_eigensolvers import QAOA, NumPyMinimumEigensolver
from qiskit_optimization.optimizers import COBYLA
from qiskit.primitives import StatevectorSampler
from qiskit_optimization.algorithms import CobylaOptimizer, MinimumEigenOptimizer
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.translators import from_docplex_mp
from qiskit_optimization.algorithms.admm_optimizer import ADMMParameters, ADMMOptimizer

# If CPLEX is installed, you can uncomment this line to import the CplexOptimizer.
# CPLEX can be used in this tutorial to solve the convex continuous problem,
# but also as a reference to solve the QUBO, or even the full problem.
#
from qiskit_optimization.algorithms import CplexOptimizer

In [ ]:
# Initialize all the algorithms we plan to use later in this tutorial.
#To solve the QUBO problems we can choose between 
#a)- MinimumEigenOptimizer using different MinimumEigensolver, such as VQE, QAOA or 
#b)NumpyMinimumEigensolver (classical) - GroverOptimizer - CplexOptimizer (classical, if CPLEX is installed)

#To solve the convex continuous problems we can choose between 
#the following classical solvers: - CplexOptimizer (if CPLEX is installed) - CobylaOptimizer
#In case CPLEX is not available, the CobylaOptimizer (for convex continuous problems) and 
#the MinimumEigenOptimizer using the NumpyMinimumEigensolver (for QUBOs) 
#can be used as classical alternatives to CPLEX for testing, validation, and benchmarking.

# define COBYLA optimizer to handle convex continuous problems.
cobyla = CobylaOptimizer()

# define QAOA via the minimum eigen optimizer
qaoa = MinimumEigenOptimizer(QAOA(sampler=StatevectorSampler(), optimizer=COBYLA()))

# exact QUBO solver as classical benchmark
exact = MinimumEigenOptimizer(NumPyMinimumEigensolver()) # to solve QUBOs

# in case CPLEX is installed it can also be used for the convex problems, the QUBO,
# or as a benchmark for the full problem.
#
#cplex = CplexOptimizer()

In [ ]:
#We test 3-ADMM-H algorithm 
#on a simple Mixed-Binary Quadratic Problem with equality and inequality constraints 

#We first construct a docplex problem and then load it into a QuadraticProgram.

# construct model using docplex
mdl = Model('AMC-3ADMM-H')

v = mdl.binary_var(name='v')
w = mdl.binary_var(name='w')
t = mdl.binary_var(name='t')
u = mdl.continuous_var(name='u')

mdl.minimize(v + w + t + 5 * (u-2)**2)
mdl.add_constraint(v + 2 * w + t + u <= 3, "cons1")
mdl.add_constraint(v + w + t >= 1, "cons2")
mdl.add_constraint(v + w == 1, "cons3")

# load quadratic program from docplex model
qp = from_docplex_mp(mdl)
print(qp.export_as_lp_string())
qp.write_to_lp_file('optimization\\problemFiles\\abc.lp')

## Classical Solution

3-ADMM-H needs <br>
a) a QUBO optimizer to solve the QUBO subproblem, and <br>
b) a continuous optimizer to solve the continuous convex constrained subproblem. 

Let us use the MinimumEigenOptimizer with the NumPyMinimumEigenSolver as a classical and exact QUBO solver 
and we use the CobylaOptimizer as a continuous convex solver. 

If CPLEX is installed, the CplexOptimizer can also be used as both, a QUBO and convex solver.

3-ADMM-H supports any other suitable solver available in Qiskit. For instance, VQE, QAOA, and GroverOptimizer can be invoked as quantum solvers, as demonstrated later. 

### Parameters
The 3-ADMM-H are wrapped in class ADMMParameters. Customized parameter values can be set as arguments of the class. In this example, parameters ρ,β are initialized to 1001 and 1000, respectively. The penalization factor_c of equality constraints Gx=b is set to 900. 

The tolerance tol for primal residual convergence is set to 1.e-6. In this case, the 3-block implementation is guaranteed to converge for Theorem 4 of [1], because the inequality constraint with the continuous variable is always active. The 2-block implementation can be run by setting three_block=False, and practically converges to a feasible not optimal solution.

In [ ]:
admm_params = ADMMParameters(
                            rho_initial=1001,
                            beta=1000,
                            factor_c=900,
                            maxiter=100,
                            three_block=True, tol=1.e-6
                        )



In [ ]:
#Calling 3-ADMM-H algorithm
#To invoke the 3-ADMM-H algorithm, an instance of the ADMMOptimizer class needs to be created. 
#This takes ADMM-specific parameters and the subproblem optimizers separately into the constructor. 
#The solution returned is an instance of OptimizationResult class.

# define QUBO optimizer
qubo_optimizer = exact
# qubo_optimizer = cplex  # uncomment to use CPLEX instead

# define classical optimizer
convex_optimizer = cobyla
# convex_optimizer = cplex  # uncomment to use CPLEX instead

# initialize ADMM with classical QUBO and convex optimizer
admm = ADMMOptimizer(params=admm_params,
                     qubo_optimizer=qubo_optimizer,
                     continuous_optimizer=convex_optimizer)

In [ ]:
# run ADMM to solve problem
result = admm.solve(qp)

In [ ]:
#Classical Solver Result
#The 3-ADMM-H solution can be then printed and visualized. 
#The x attribute of the solution contains respectively, the values of the binary decision variables and 
#the values of the continuous decision variables. 
#The fval is the objective value of the solution.
print("x={}".format(result.x))
print("fval={:.2f}".format(result.fval))

In [ ]:
#Solution statistics
#can be accessed in the state field and visualized. 
#We here display the convergence of 3-ADMM-H, in terms of primal residuals.

plt.plot(result.state.residuals)
plt.xlabel("Iterations")
plt.ylabel("Residuals")
plt.show()

## Quantum Solution

We now solve the same optimization problem with QAOA as QUBO optimizer, running on simulated quantum device. 
First, one need to select the classical optimizer of the eigensolver QAOA. Then, the simulation backend is set. Finally, the eigensolver is wrapped into the MinimumEigenOptimizer class. A new instance of ADMMOptimizer is populated with QAOA as QUBO optimizer.

In [ ]:
# define QUBO optimizer
qubo_optimizer = qaoa

# define classical optimizer
convex_optimizer = cobyla
# convex_optimizer = cplex  # uncomment to use CPLEX instead

# initialize ADMM with quantum QUBO optimizer and classical convex optimizer
admm_q = ADMMOptimizer(params=admm_params,
                       qubo_optimizer=qubo_optimizer,
                       continuous_optimizer=convex_optimizer)

In [ ]:
# run ADMM to solve problem
result_q = admm_q.solve(qp)

In [ ]:
#Quantum Solver Results
#x stands for the solution, the fval is for objective value.
print("x={}".format(result_q.x))
print("fval={:.2f}".format(result_q.fval))

In [ ]:
plt.clf()
plt.plot(result_q.state.residuals)
plt.xlabel("Iterations")
plt.ylabel("Residuals")
plt.show()